In [16]:
from speechmos import dnsmos, plcmos
from apache_beam.ml.inference.base import KeyedModelHandler
from apache_beam.ml.inference.base import PredictionResult
from apache_beam.ml.inference.base import RunInference
from apache_beam.ml.inference.pytorch_inference import PytorchModelHandlerTensor
from apache_beam.ml.inference.pytorch_inference import PytorchModelHandlerKeyedTensor
from apache_beam.options.pipeline_options import PipelineOptions
import torchaudio
import torchaudio.functional as F
from torchcodec.decoders import AudioDecoder
from torchcodec import AudioSamples
import numpy as np
import io
import soundfile as sf
import pandas as pd
import uuid
from typing import Dict, Any, Iterable, Sequence, Tuple
import torch
import torchcodec
import apache_beam as beam
from apache_beam.ml.inference.base import RunInference, ModelHandler, PredictionResult
from apache_beam.runners.interactive.interactive_runner import InteractiveRunner
import apache_beam.runners.interactive.interactive_beam as ib
from apache_beam.options import pipeline_options
import torch.nn as nn
import torchaudio.functional as F
from transformers import Wav2Vec2Model
from pathlib import Path
from transformers import Wav2Vec2Config, Wav2Vec2Model


In [17]:
df = pd.read_parquet("data/vaani.parquet")
x = df["audio"].to_list()
x = [{"audio": a} for a in x]
audios = [{"id": str(uuid.uuid4()), "audio": {"bytes": a["audio"]["bytes"]}} for a in x]
len(audios)

5

In [18]:
def decode_and_key_audio(
    element: Dict[str, Any],
) -> Tuple[str, torchcodec.AudioSamples]:
    # pre-processing
    try:
        element_id = element["id"]
        audio_bytes = element["audio"]["bytes"]
        audio_samples: torchcodec.AudioSamples = torchcodec.decoders.AudioDecoder(audio_bytes).get_all_samples()
        audio_data = audio_samples.data
        if audio_data.ndim >= 1:  # Make it 1D array Stereo -> Mono
            audio_data = audio_data.mean(axis=0)
            assert audio_data.ndim == 1
        # print(
        #     f"Shape of the audio is:{audio_data.shape} \t type of the audio is : {type(audio_data)}"
        # )
        audio_data = F.resample(audio_data, audio_samples.sample_rate, 16000)
        audio_data = torch.clamp(audio_data, min=-1, max=1)
        assert audio_data.ndim == 1 
        print(audio_samples.duration_seconds)
        audio_samples.data = audio_data
        
        # print(audio_samples)
        return (element_id, audio_samples)

    except RuntimeError as e:
        print(
            f" Could not decode audio for ID {element.get('id', 'Unknown')}. Error: {e}"
        )
        return None


def format_output(prediction_result: PredictionResult) -> Dict[str, Any]:
    # post-processing
    key, prediction = prediction_result.example, prediction_result.inference
    output_dict = {"id": key}
    output_dict.update(prediction)
    return output_dict

In [19]:
class SpeechMOSMetrics:
    """Calculates MOS scores using the speechmos library."""

    def __init__(self):
        self.dnsmos_model = dnsmos
        self.plcmos_model = plcmos

    def compute(self, audio_samples: torchcodec.AudioSamples) -> Dict[str, float]:
        """
        Computes MOS scores for the given audio samples.
        The audio should be single-channel and will be resampled to 16kHz.
        """
        # The speechmos library expects a numpy array
        audio_tensor = (
            F.resample(audio_samples.data, audio_samples.sample_rate, 16000)
            .squeeze(0)
            .cpu()
            .numpy()
        )
        # print(audio_tensor.shape)
        if audio_tensor.ndim > 1 :
            audio_tensor = audio_tensor.mean(axis=0)
            assert audio_tensor.ndim==1
        sample_rate = 16000
        audio_tensor = audio_tensor/np.linalg.norm(audio_tensor, ord=2)
        dnsmos_score = self.dnsmos_model.run(audio_tensor, sample_rate)
        plcmos_score = self.plcmos_model.run(audio_tensor, sample_rate)
        
        scores = {}
        if isinstance(dnsmos_score, dict):
            scores.update(
                {
                    "DNSMOS_OVRL_MOS": dnsmos_score.get("ovrl_mos", 0.0),
                    "DNSMOS_SIG_MOS": dnsmos_score.get("sig_mos", 0.0),
                    "DNSMOS_BAK_MOS": dnsmos_score.get("bak_mos", 0.0),
                    "DNSMOS_P808_MOS": dnsmos_score.get("p808_mos", 0.0),
                }
            )
        if isinstance(plcmos_score, dict):
            scores.update({"PLCMOS_MOS": plcmos_score.get("plcmos", 0.0)})
        return scores

In [20]:
class SquimMetrics:
    """Calculates SQUIM metrics using a pre-trained torchaudio model."""

    def __init__(self, model_path: str, device: str = "cpu"):
        self.model = torchaudio.models.squim_objective_base()
        state_dict = torch.load(model_path, map_location=torch.device(device))
        self.model.load_state_dict(state_dict)
        self.device = device
        self.model = self.model.to(self.device)
        self.model = self.model.eval()

    def compute(self, audio_samples: torchcodec.AudioSamples) -> Dict[str, float]:
        """Computes STOI, PESQ, and SI-SDR scores."""
        audio_tensor = audio_samples.data
        sample_rate = audio_samples.sample_rate
        if sample_rate != 16000:
            audio_tensor = F.resample(audio_tensor, sample_rate, 16000)
        # Convert Mono or Unsqueeze
        if audio_tensor.ndim == 1:
            audio_tensor = audio_tensor.unsqueeze(0)
        audio_tensor = audio_tensor.to(self.device)
        stoi, pesq, si_sdr = self.model(audio_tensor)
        return {
            "STOI": stoi.item(),
            "PESQ": pesq.item(),
            "SI_SDR": si_sdr.item(),
        }

In [21]:
class VADMetrics:
    """Calculates Voice Activity Detection metrics using Silero-VAD."""

    def __init__(self):
        self.model, utils = torch.hub.load(
            repo_or_dir="snakers4/silero-vad", model="silero_vad"
        )
        (self.get_speech_timestamps, _, _, _, _) = utils

    def compute(self, audio_samples: torchcodec.AudioSamples) -> Dict[str, float]:
        """Computes silence ratio and non-audio silence metrics."""
        audio_tensor = audio_samples.data
        sample_rate = audio_samples.sample_rate
        if sample_rate != 16000:
            audio_tensor = F.resample(audio_tensor, sample_rate, 16000)

        # audio_tensor = audio_tensor
        if audio_tensor.ndim == 2:
            audio_tensor = audio_tensor.squeeze(0)
        else:
            ValueError("Shape of the audio should be 1D tensor")
        duration = audio_samples.duration_seconds

        speech_timestamps = self.get_speech_timestamps(
            audio_tensor,
            self.model,
            return_seconds=True,
        )
        # print(speech_timestamps)
        silence_ratio = self.calculate_total_silence_ratio(speech_timestamps, duration)
        non_audio_silence = self.non_audio_silence(speech_timestamps, duration)
        return {
            "SILENCE_RATIO": silence_ratio,
            "NON_AUDIO_SILENCE": non_audio_silence,
        }

    @staticmethod
    def calculate_total_silence_ratio(segments, total_duration):
        if not segments or total_duration == 0:
            return 0.0  # If no speech detected, it's all silence
        speech_time = sum(seg["end"] - seg["start"] for seg in segments)
        silence_time = total_duration - speech_time
        return silence_time / total_duration

    @staticmethod
    def non_audio_silence(speech_timestamps, total_duration):
        if not speech_timestamps:
            return 0  # If no speech, all duration is non-audio silence

        timestamps = []
        for tm in speech_timestamps:
            for val in tm.values():
                timestamps.append(val)

        # Silence at the beginning + silence at the end
        return timestamps[0] + (total_duration - timestamps[-1])

In [22]:
class CombinedMetricsModelHandler(
    ModelHandler[Tuple[str, torchcodec.AudioSamples], PredictionResult, Any]
):
    def __init__(self, squim_model_path: str, device: str = "gpu"):
        self._squim_model_path = squim_model_path
        self._device = device
        self.mos_model = None
        self.squim_model = None
        self.vad_model = None

    def load_model(self) -> Dict[str, Any]:
        self.mos_model = SpeechMOSMetrics()
        self.squim_model = SquimMetrics(self._squim_model_path, device=self._device)
        self.vad_model = VADMetrics()
        return {"mos": self.mos_model, "squim": self.squim_model, "vad": self.vad_model}

    def run_inference(
        self,
        batch: Sequence[Tuple[str, torchcodec.AudioSamples]],
        models: Dict[str, Any],
        inference_args: Dict[str, Any] | None = None,
    ) -> Iterable[PredictionResult]:
        for key, audio_samples in batch:
            mos_scores = self.mos_model.compute(audio_samples)
            squim_scores = self.squim_model.compute(audio_samples)
            vad_scores = self.vad_model.compute(audio_samples)

            combined_scores = {**mos_scores, **squim_scores, **vad_scores}
            yield PredictionResult(example=key, inference=combined_scores)

    def share_model_across_processes(self) -> bool:
        return True

    def model_copies(self) -> int:
        return 1

In [23]:
class TripletModel(nn.Module):
    def __init__(self, ssl_out_dim=768, emb_dim=256):
        super(TripletModel, self).__init__()
        # Initialize wav2vec from a default config.
        # The trained weights will be loaded from the state_dict later.
        config = Wav2Vec2Config()
        model = Wav2Vec2Model(config)
        self.ssl_model = model
        self.ssl_features = ssl_out_dim
        self.embedding_layer = nn.Sequential(
            nn.ReLU(), nn.Linear(self.ssl_features, emb_dim)
        )

    def forward(self, wav, phead=False):
        if wav.ndim == 1:
            wav = wav.unsqueeze(0)
        if wav.ndim > 2:
            wav = wav.squeeze(1)
        res = self.ssl_model(wav).last_hidden_state
        x = torch.mean(res, 1)
        if phead:
            x = self.embedding_layer(x)
        x = torch.nn.functional.normalize(x, dim=1)
        return x

class MosPredictor(nn.Module):
    def __init__(self, emb_dim=768):
        super(MosPredictor, self).__init__()
        # TripletModel is initialized internally, as you specified.
        self.pt_model = TripletModel()
        self.mos_layer = nn.Linear(emb_dim, 1)

    def forward(self, wav):
        x = self.pt_model(wav, phead=False)
        if len(x.shape) == 3:
            x.squeeze_(2)
        out = self.mos_layer(x)
        return out

class ScoreQMetrics:
    def __init__(self, model_path: str, device: str = "cpu"):
        self.device = torch.device(device)

        self.model = MosPredictor()

        state_dict = torch.load(model_path, map_location=self.device)
        self.model.load_state_dict(state_dict)
        
        self.model.to(self.device)
        self.model.eval()
        print("ScoreQ model loaded successfully.")

    def compute(self, audio_samples: torchcodec.AudioSamples) -> Dict[str, float]:
        """Computes the ScoreQ MOS score."""
        audio_tensor = audio_samples.data.float()
        sample_rate = audio_samples.sample_rate

        if sample_rate != 16000:
            audio_tensor = F.resample(audio_tensor, sample_rate, 16000)

        if audio_tensor.ndim > 1:
            audio_tensor = torch.mean(audio_tensor, dim=0, keepdim=True)
        else:
            audio_tensor = audio_tensor.unsqueeze(0)

        audio_tensor = audio_tensor.to(self.device)

        with torch.no_grad():
            mos_tensor = self.model(audio_tensor)

        mos_score = mos_tensor.item()
        return {"SCOREQ_MOS": mos_score}

In [24]:
class ScoreQModelHandler(ModelHandler[Tuple[str, torchcodec.AudioSamples], PredictionResult, Any]):
    def __init__(self, scoreq_model_path: str, device: str = "gpu"):
        self._scoreq_model_path = scoreq_model_path
        self._device = device
        self.scoreq_model = None

    def load_model(self) -> None:
        self.scoreq_model = ScoreQMetrics(
            model_path=self._scoreq_model_path,
            device=self._device
        )

    def run_inference(
        self,
        batch: Sequence[Tuple[str, torchcodec.AudioSamples]],
        model: Any,
        inference_args: Dict[str, Any] | None = None,
    ) -> Iterable[PredictionResult]:
        for key, audio_samples in batch:
            scoreq_scores = self.scoreq_model.compute(audio_samples)
            yield PredictionResult(example=key, inference=scoreq_scores)


In [25]:
def merge_metrics(element: Tuple[str, Dict[str, list]]) -> Dict[str, Any]:
    key, grouped_data = element
    combined_dict = grouped_data['combined'][0]
    scoreq_dict = grouped_data['scoreq'][0]
    del scoreq_dict['id']
    final_dict = {**combined_dict, **scoreq_dict}
    return final_dict


In [26]:
class RunAudioInference(beam.PTransform):
    def __init__(self, combined_handler: ModelHandler, scoreq_handler: ModelHandler):
        self.combined_handler = combined_handler
        self.scoreq_handler = scoreq_handler

    def expand(self, pcoll: beam.PCollection) -> beam.PCollection:
        keyed_audio_samples = (
            pcoll
            | "DecodeAudio" >> beam.Map(decode_and_key_audio)
            | "FilterValid" >> beam.Filter(lambda x: x is not None)
        )
        combined_results = (
            keyed_audio_samples
            | "RunCombinedInference" >> RunInference(self.combined_handler)
            | "FormatCombinedOutput" >> beam.Map(format_output)
            | "KeyCombinedResults" >> beam.Map(lambda x: (x['id'], x))
        )
        scoreq_results = (
            keyed_audio_samples
            | "RunScoreQInference" >> RunInference(self.scoreq_handler)
            | "FormatScoreQOutput" >> beam.Map(format_output)
            | "KeyScoreQResults" >> beam.Map(lambda x: (x['id'], x))
        )
        all_results = (
            {'combined': combined_results, 'scoreq': scoreq_results}
            | "GroupByID" >> beam.CoGroupByKey()
            | "MergeMetrics" >> beam.Map(merge_metrics)
        )
        return all_results

In [27]:
SCOREQ_MODEL_PATH = "/home/jupyter/notebooks/scoreq_model.pt"
SQUIM_MODEL_PATH = "/home/jupyter/notebooks/squim_objective_dns2020.pth"

combined_mh = CombinedMetricsModelHandler(
    squim_model_path=SQUIM_MODEL_PATH, device="cuda"
)

scoreq_mh = ScoreQModelHandler(
    scoreq_model_path=SCOREQ_MODEL_PATH,
    device="cuda"
)


In [28]:
with beam.Pipeline(InteractiveRunner(), options=PipelineOptions()) as p:
    output = (
        p
        | "CreateSingleSample" >> beam.Create(audios)
        # Pass both handlers to the PTransform
        | "CalculateSpeechMetrics" >> RunAudioInference(
            combined_handler=combined_mh,
            scoreq_handler=scoreq_mh
        )
    )

Using cache found in /home/jupyter/.cache/torch/hub/snakers4_silero-vad_master


ScoreQ model loaded successfully.
2.864
4.156
3.036
10.75
4.386


In [29]:
ib.show(output)